# Legal_HF Retriever Notebook (FAISS + Evaluation)
Notebook này giúp bạn:
1) Encode chunks bằng **Quockhanh05/Vietnam_legal_embeddings** (Legal_HF)
2) Build **FAISS IndexFlatIP** (cosine)
3) Đánh giá **Recall@k** và **MRR@10** trên `eval_qa.jsonl`

**Bạn chỉ cần sửa PATH ở Cell 2.**

In [ ]:
# Cell 1 — Install deps (Colab/Local)
# Nếu chạy local và đã cài sẵn thì có thể bỏ cell này.
!pip -q install -U sentence-transformers faiss-cpu numpy pandas tqdm


In [ ]:
# Cell 2 — Config paths
import os, json, time
import numpy as np
import pandas as pd

# TODO: sửa đường dẫn cho đúng dự án của bạn
CHUNKS_PATH = r"./output_nghidinh/chunks_clean.json"   # hoặc chunks.json
EVAL_QA_PATH = r"./eval_qa.jsonl"                      # file eval của bạn

OUT_DIR = r"./outputs_legal_hf"
os.makedirs(OUT_DIR, exist_ok=True)

print("CHUNKS_PATH exists:", os.path.exists(CHUNKS_PATH))
print("EVAL_QA_PATH exists:", os.path.exists(EVAL_QA_PATH))
print("OUT_DIR:", OUT_DIR)


In [ ]:
# Cell 3 — Load chunks + eval set
chunks = json.load(open(CHUNKS_PATH, "r", encoding="utf-8"))
passages = [c["text"] for c in chunks]
meta = [c.get("metadata", {}) for c in chunks]

print("Total chunks:", len(chunks))
print("Example chunk keys:", chunks[0].keys())
print("Example meta keys:", (chunks[0].get("metadata") or {}).keys())

eval_items = []
with open(EVAL_QA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)

        q = obj.get("question") or obj.get("query")
        exp = None
        if "expected_chunk_index" in obj:
            exp = obj["expected_chunk_index"]
        else:
            cits = obj.get("expected_citations", [])
            if cits and isinstance(cits, list) and "chunk_index" in cits[0]:
                exp = cits[0]["chunk_index"]

        if q and exp is not None:
            eval_items.append((q, int(exp)))

print("Eval items:", len(eval_items))
print("First eval item:", eval_items[0] if eval_items else None)


In [ ]:
# Cell 4 — Encode chunks with Legal_HF
import torch
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "Quockhanh05/Vietnam_legal_embeddings"
model = SentenceTransformer(model_name, device=device)

t0 = time.perf_counter()
emb_p = model.encode(
    passages,
    batch_size=64 if device=="cuda" else 16,
    show_progress_bar=True,
    normalize_embeddings=True
)
emb_p = np.asarray(emb_p, dtype="float32")
t_build = time.perf_counter() - t0

print("Embeddings shape:", emb_p.shape)
print("Build embeddings sec:", round(t_build, 2))

np.save(os.path.join(OUT_DIR, "legal_hf_embeddings.npy"), emb_p)
with open(os.path.join(OUT_DIR, "legal_hf_metadata.json"), "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print("Saved embeddings & metadata to:", OUT_DIR)


In [ ]:
# Cell 5 — Build FAISS cosine index (IndexFlatIP)
import faiss

dim = emb_p.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(emb_p)

faiss_path = os.path.join(OUT_DIR, "legal_hf_faiss.index")
faiss.write_index(index, faiss_path)
print("FAISS index saved:", faiss_path)


In [ ]:
# Cell 6 — Metrics helpers
def recall_at_k(ranks, k):
    import numpy as np
    return float(np.mean([1.0 if r <= k else 0.0 for r in ranks]))

def mrr_at_k(ranks, k):
    import numpy as np
    rr = []
    for r in ranks:
        rr.append(1.0 / r if r <= k else 0.0)
    return float(np.mean(rr))


In [ ]:
# Cell 7 — Evaluate Retriever (Legal_HF + FAISS)
import time, numpy as np
from tqdm import tqdm

topN = 50
ranks = []
encode_times = []
search_times = []

for q, gt in tqdm(eval_items, desc="Evaluate Legal_HF"):
    t1 = time.perf_counter()
    qv = model.encode([q], normalize_embeddings=True)
    encode_times.append(time.perf_counter() - t1)

    qv = np.asarray(qv, dtype="float32")

    t2 = time.perf_counter()
    scores, ids = index.search(qv, topN)
    search_times.append(time.perf_counter() - t2)

    retrieved = ids[0].tolist()
    if gt in retrieved:
        rank = retrieved.index(gt) + 1
    else:
        rank = 10**9
    ranks.append(rank)

metrics = {
    "model": model_name,
    "topN": topN,
    "Recall@1": round(recall_at_k(ranks, 1), 4),
    "Recall@3": round(recall_at_k(ranks, 3), 4),
    "Recall@5": round(recall_at_k(ranks, 5), 4),
    "MRR@10": round(mrr_at_k(ranks, 10), 4),
    "avg_encode_ms": round(1000*np.mean(encode_times), 3),
    "avg_search_ms": round(1000*np.mean(search_times), 3),
}

metrics


In [ ]:
# Cell 8 — Save metrics table
import pandas as pd, os
df = pd.DataFrame([metrics])
csv_path = os.path.join(OUT_DIR, "legal_hf_retriever_metrics.csv")
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print("Saved:", csv_path)
df


## Notes
- Nếu `eval_qa.jsonl` của bạn không có `chunk_index` mà chỉ có `{van_ban,dieu,khoan}`, bạn cần map expected_citations → chunk_index trước khi đánh giá.
- Nếu muốn so sánh PhoBERT vs Legal_HF, chạy notebook tương tự cho PhoBERT và ghép 2 bảng metrics.
